In [1]:
import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()
uploaded_names = list(uploaded.keys())

planned_file = next(
    name for name in uploaded_names
    if "planned_bim_integrated" in name.lower()
)

asbuilt_file = next(
    name for name in uploaded_names
    if "asbuilt_bim_integrated" in name.lower()
)

planned = pd.read_csv(planned_file)
asbuilt = pd.read_csv(asbuilt_file)

print("Planned:", planned.shape)
print("As-built:", asbuilt.shape)

Saving asbuilt_bim_integrated.csv to asbuilt_bim_integrated.csv
Saving planned_bim_integrated.csv to planned_bim_integrated.csv
Planned: (7166, 24)
As-built: (3661, 25)


In [5]:
planned_duplicates = planned.duplicated(
    subset=["GUID", "TaskID"]
).sum()

asbuilt_duplicates = asbuilt.duplicated(
    subset=["GUID", "TaskID"]
).sum()

print(
    "Duplicate planned GUID–TaskID combinations:",
    planned_duplicates
)

print(
    "Duplicate as-built GUID–TaskID combinations:",
    asbuilt_duplicates
)

Duplicate planned GUID–TaskID combinations: 0
Duplicate as-built GUID–TaskID combinations: 0


In [6]:
# Select only observed-status information from as-built data
observed_status = asbuilt[
    [
        "GUID",
        "TaskID",
        "Element on time or not"
    ]
].copy()

# Merge observed status into every planned element-task record
dashboard_master = planned.merge(
    observed_status,
    on=["GUID", "TaskID"],
    how="left",
    validate="one_to_one"
)

# Create an observation indicator
dashboard_master["ObservedFlag"] = (
    dashboard_master[
        "Element on time or not"
    ].notna()
).astype(int)

# Create a clear dashboard status
dashboard_master["ProgressStatus"] = (
    dashboard_master[
        "Element on time or not"
    ]
    .fillna("Not represented in as-built log")
)

# LateFlag remains missing for unobserved records
dashboard_master["LateFlag"] = pd.Series(
    pd.NA,
    index=dashboard_master.index,
    dtype="Int64"
)

dashboard_master.loc[
    dashboard_master["ObservedFlag"] == 1,
    "LateFlag"
] = (
    dashboard_master.loc[
        dashboard_master["ObservedFlag"] == 1,
        "Element on time or not"
    ] == "Too late"
).astype(int)

# Create a unique event key
dashboard_master["EventKey"] = (
    dashboard_master["GUID"]
    + "|"
    + dashboard_master["TaskID"]
)

# Date fields useful in Power BI
dashboard_master["PlannedStartMonth"] = (
    dashboard_master["TaskStart"]
    .dt.to_period("M")
    .astype(str)
)

dashboard_master["PlannedFinishMonth"] = (
    dashboard_master["TaskFinish"]
    .dt.to_period("M")
    .astype(str)
)

print("Master-table records:", len(dashboard_master))
print("Unique event keys:", dashboard_master["EventKey"].nunique())

print("\nProgress status:")
print(
    dashboard_master["ProgressStatus"]
    .value_counts()
)

print(
    "\nObserved records:",
    dashboard_master["ObservedFlag"].sum()
)

print(
    "Late observed records:",
    dashboard_master["LateFlag"].sum()
)

Master-table records: 7166
Unique event keys: 7166

Progress status:
ProgressStatus
Not represented in as-built log    3505
On time                            3467
Too late                            194
Name: count, dtype: int64

Observed records: 3661
Late observed records: 194


In [7]:
# Select one BIM record for each unique element
element_columns = [
    "GUID",
    "Canonical_IfcClass",
    "BIM_IfcClass",
    "BIM_ElementName",
    "BIM_Description",
    "BIM_ObjectType",
    "BIM_Tag",
    "BIM_TypeName",
    "BIM_BuildingStorey",
    "BIM_HasGeometry"
]

# Use only columns that exist
element_columns = [
    column for column in element_columns
    if column in dashboard_master.columns
]

dim_elements = (
    dashboard_master[element_columns]
    .drop_duplicates(subset="GUID")
    .copy()
)

# Calculate element-level event statistics
element_statistics = (
    dashboard_master.groupby("GUID")
    .agg(
        Planned_Task_Count=(
            "TaskID",
            "nunique"
        ),
        Observed_Task_Count=(
            "ObservedFlag",
            "sum"
        ),
        Late_Task_Count=(
            "LateFlag",
            "sum"
        )
    )
    .reset_index()
)

dim_elements = dim_elements.merge(
    element_statistics,
    on="GUID",
    how="left",
    validate="one_to_one"
)

dim_elements["Task_Observation_Coverage"] = (
    dim_elements["Observed_Task_Count"]
    / dim_elements["Planned_Task_Count"]
    * 100
).round(2)

dim_elements["Ever_Observed"] = (
    dim_elements["Observed_Task_Count"] > 0
).astype(int)

dim_elements["Ever_Late"] = (
    dim_elements["Late_Task_Count"] > 0
).astype(int)

dim_elements["Element_Status"] = np.select(
    [
        dim_elements["Ever_Late"] == 1,

        (
            dim_elements["Observed_Task_Count"] > 0
        )
        & (
            dim_elements["Observed_Task_Count"]
            < dim_elements["Planned_Task_Count"]
        ),

        (
            dim_elements["Observed_Task_Count"]
            == dim_elements["Planned_Task_Count"]
        ),

        dim_elements["Observed_Task_Count"] == 0
    ],
    [
        "Affected by delay",
        "Partially observed",
        "Fully observed without late record",
        "Not represented in as-built log"
    ],
    default="Review required"
)

print("Element dimension:", dim_elements.shape)

print(
    dim_elements["Element_Status"]
    .value_counts()
)

Element dimension: (3505, 17)
Element_Status
Partially observed                 2239
Not represented in as-built log    1147
Affected by delay                   119
Name: count, dtype: int64


In [8]:
# Planned activity information
dim_tasks = (
    dashboard_master.groupby(
        ["TaskID", "TaskName"],
        dropna=False
    )
    .agg(
        Planned_Start=("TaskStart", "min"),
        Planned_Finish=("TaskFinish", "max"),
        Planned_Records=("EventKey", "nunique"),
        Planned_Elements=("GUID", "nunique")
    )
    .reset_index()
)

# Observed information by activity
observed_tasks = (
    dashboard_master[
        dashboard_master["ObservedFlag"] == 1
    ]
    .groupby("TaskID")
    .agg(
        Observed_Records=("EventKey", "nunique"),
        Observed_Elements=("GUID", "nunique")
    )
    .reset_index()
)

# Late information by activity
late_tasks = (
    dashboard_master[
        dashboard_master["LateFlag"] == 1
    ]
    .groupby("TaskID")
    .agg(
        Late_Records=("EventKey", "nunique"),
        Delayed_Elements=("GUID", "nunique")
    )
    .reset_index()
)

dim_tasks = (
    dim_tasks
    .merge(
        observed_tasks,
        on="TaskID",
        how="left"
    )
    .merge(
        late_tasks,
        on="TaskID",
        how="left"
    )
)

numeric_columns = [
    "Observed_Records",
    "Observed_Elements",
    "Late_Records",
    "Delayed_Elements"
]

dim_tasks[numeric_columns] = (
    dim_tasks[numeric_columns]
    .fillna(0)
    .astype(int)
)

dim_tasks["Observation_Coverage_Percent"] = (
    dim_tasks["Observed_Elements"]
    / dim_tasks["Planned_Elements"]
    * 100
).round(2)

dim_tasks["Observed_Delay_Rate_Percent"] = (
    dim_tasks["Delayed_Elements"]
    / dim_tasks["Observed_Elements"]
    .replace(0, np.nan)
    * 100
).round(2)

dim_tasks["Planned_Duration_Days"] = (
    dim_tasks["Planned_Finish"]
    - dim_tasks["Planned_Start"]
).dt.days + 1

dim_tasks = dim_tasks.sort_values(
    ["Planned_Start", "TaskID"]
).reset_index(drop=True)

print("Task dimension:", dim_tasks.shape)

display(
    dim_tasks.sort_values(
        "Late_Records",
        ascending=False
    ).head(10)
)

Task dimension: (73, 13)


,TaskID,TaskName,Planned_Start,Planned_Finish,Planned_Records,Planned_Elements,Observed_Records,Observed_Elements,Late_Records,Delayed_Elements,Observation_Coverage_Percent,Observed_Delay_Rate_Percent,Planned_Duration_Days
49,ST01060,Metselwerk,2015-07-09,2015-07-21,527,527,527,527,60,60,100.0,11.39,13
46,ST00980,Plaatsen kozijnen,2015-07-06,2015-07-10,171,171,171,171,50,50,100.0,29.24,5
45,ST00950,Stelwerk buitengevel,2015-07-02,2015-07-08,171,171,171,171,50,50,100.0,29.24,7
59,ST01070,Metselwerk,2015-07-22,2015-08-04,199,199,199,199,13,13,100.0,6.53,14
60,ST01000,Prefab betonband,2015-08-17,2015-08-18,84,84,84,84,10,10,100.0,11.90,2
38,ST00970,Plaatsen kozijnen,2015-06-18,2015-06-24,176,176,176,176,7,7,100.0,3.98,7
63,ST01130,Plaatsen scharnierkap incl. goot en platte daken,2015-08-28,2015-09-17,313,313,313,313,2,2,100.0,0.64,21
39,ST00510,Lijmwerk kalkzandsteen elementen,2015-06-19,2015-06-23,19,19,19,19,2,2,100.0,10.53,5
0,ST00060,Stellen bruggetjes,2015-02-26,2015-02-27,1,1,1,1,0,0,100.0,0.00,2
8,ST00150,Stort liftwanden,2015-03-10,2015-03-10,17,17,17,17,0,0,100.0,0.00,1


In [9]:
dim_tasks["Task_Coverage_Status"] = np.select(
    [
        dim_tasks[
            "Observation_Coverage_Percent"
        ] == 100,

        dim_tasks[
            "Observation_Coverage_Percent"
        ] == 0
    ],
    [
        "Fully observed",
        "Not represented in as-built log"
    ],
    default="Partially observed"
)

dim_tasks["Task_Delay_Status"] = np.where(
    dim_tasks["Late_Records"] > 0,
    "Contains late elements",
    "No late elements recorded"
)

print("Task coverage:")
print(
    dim_tasks["Task_Coverage_Status"]
    .value_counts()
)

print("\nTask delay status:")
print(
    dim_tasks["Task_Delay_Status"]
    .value_counts()
)

Task coverage:
Task_Coverage_Status
Fully observed                     72
Not represented in as-built log     1
Name: count, dtype: int64

Task delay status:
Task_Delay_Status
No late elements recorded    65
Contains late elements        8
Name: count, dtype: int64


In [10]:
print("Element dimension:", dim_elements.shape)

print("\nElement status:")
print(
    dim_elements["Element_Status"]
    .value_counts()
)

Element dimension: (3505, 17)

Element status:
Element_Status
Partially observed                 2239
Not represented in as-built log    1147
Affected by delay                   119
Name: count, dtype: int64


In [12]:
assert dashboard_master["EventKey"].is_unique
assert dim_elements["GUID"].is_unique
assert dim_tasks["TaskID"].is_unique

print("All Power BI keys are unique.")

All Power BI keys are unique.


In [13]:
dashboard_master.to_csv(
    "Fact_ProgressEvents.csv",
    index=False
)

dim_elements.to_csv(
    "Dim_Elements.csv",
    index=False
)

dim_tasks.to_csv(
    "Dim_Tasks.csv",
    index=False
)

print("Files created successfully.")

Files created successfully.


In [14]:
from google.colab import files

files.download("Fact_ProgressEvents.csv")
files.download("Dim_Elements.csv")
files.download("Dim_Tasks.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>